# Método de Newton aplicado a SINGX

## Gradiente y Hessiana de SINGX

Para cada bloque $j=1,\dots,m$, con $y^{(j)}=(x_{4j-3},x_{4j-2},x_{4j-1},x_{4j})^t$, definimos
$$u^{(j)}=A\,y^{(j)},\qquad A=\begin{pmatrix}1&10&0&0\\0&0&1&-1\\0&1&-2&0\\1&0&0&-1\end{pmatrix},$$
es decir, $u_1=x_{4j-3}+10x_{4j-2}$, $u_2=x_{4j-1}-x_{4j}$, $u_3=x_{4j-2}-2x_{4j-1}$, $u_4=x_{4j-3}-x_{4j}$.

**Gradiente.** Se aplica la regla de la cadena (para más detalle ver el archivo 01_max_descenso_SINGX)

**Hessiana.** Como $\nabla_u^2\phi=\mathrm{diag}(2,\,10,\,12u_3^2,\,120u_4^2)$, se tiene
$$\nabla_{y^{(j)}}^2 f=A^t\,\mathrm{diag}(2,\,10,\,12u_3^2,\,120u_4^2)\,A.$$
La Hessiana global es bloque diagonal, con un bloque $4\times 4$ por cada $j$.

## Caso 1: Hessiana singular en $x_0=(1,1,1,1)^t$

En $x_0=(1,1,1,1)^t$ se tiene $u=(11,0,-1,0)$ y por tanto $u_4=0$, lo que anula la cuarta entrada de la diagonal $\mathrm{diag}(2,10,12,0)$ y hace que $\nabla^2 f(x_0)$ tenga rango $3<4$. **El método de Newton no está bien definido en este punto** (el sistema $\nabla^2 f(x_0)\,d_0=-\nabla f(x_0)$ no tiene solución única). Esto motiva el uso de un punto inicial alternativo o de regularización.

## Caso 2: Hessiana no singular en $x_0=(1,1,1,0)^t$

En $x_0=(1,1,1,0)^t$: $u=(11,1,-1,1)$, $u_4=1\neq 0$, luego $\nabla^2 f(x_0)$ es definida positiva (rango completo). Se tiene
$$f(x_0)=11^2+5\cdot 1^2+(-1)^4+10\cdot 1^4=137,$$
$$\nabla f(x_0)=(62,\,216,\,18,\,-50)^t,\qquad \|\nabla f(x_0)\|\approx 230.92,$$
$$\nabla^2 f(x_0)=\begin{pmatrix}122&20&0&-120\\20&212&-24&0\\0&-24&58&-10\\-120&0&-10&130\end{pmatrix}.$$
Se resuelve $\nabla^2 f(x_0)\,d_0=-\nabla f(x_0)$ y se actualiza $x_1=x_0+d_0$. La convergencia es cuadrática localmente (Teorema 9.3.1). En las proximidades de $x^*=0$, la Hessiana vuelve a ser singular (rango $2m$), por lo que el método pierde su garantía teórica y se requiere regularización $[\nabla^2 f(x_k)+\varepsilon_k I]$.

In [2]:
# Imports
import sys
from pathlib import Path

# Sube un nivel desde la carpeta del notebook hasta la raíz del repositorio
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from funciones import singx_f, singx_grad, singx_hess, newton_method

## Método de Newton para SINGX

El algoritmo se implementa numéricamente resolviendo el sistema lineal de Newton en cada iteración. Dado $x_0\in\mathbb{R}^n$ y tolerancia $\varepsilon>0$:

1. Calcular $g_k=\nabla f(x_k)$. Si $\|g_k\|<\varepsilon$, parar.
2. Resolver el sistema $\nabla^2 f(x_k)\,d_k = -g_k$ para obtener la dirección de Newton $d_k$.
3. Actualizar $x_{k+1}=x_k+d_k$ (paso unitario, $\alpha_k=1$).
4. $k\leftarrow k+1$, volver al paso 1.

**Nota sobre SINGX.** Lejos de $x^*=0$, la matriz Hessiana $\nabla^2 f(x_k)$ es definida positiva, lo que garantiza que $d_k$ es dirección de descenso y la convergencia cuadrática local (Teorema 9.3.1). Como ya se vio, la Hessiana es **singular** en el óptimo $x^*=0$ (rango $2m<n$) y también en puntos donde $u_3=u_4=0$ entonces el método de Newton no está bien definido. Por ello, en la práctica se parte de un punto inicial con Hessiana no singular (por ejemplo $x_0=(1,1,1,0)^t$) y se detiene el método cuando la Hessiana se vuelve mal condicionada cerca de $x^*=0$, o se recurre a la regularización $[\nabla^2 f(x_k)+\varepsilon_k I]$.

In [3]:
# ----- Caso 1: Hessiana singular en x0 = (1,1,1,1) -----
x0 = np.array([1., 1., 1., 1.])
print(f"Caso 1: x0 = {x0}")
print(f"f(x0)      = {singx_f(x0):.4f}")          # esperado: 122.0
print(f"||grad||   = {np.linalg.norm(singx_grad(x0)):.4f}")  # esperado: ≈ 217.26
print(f"rang(H)    = {np.linalg.matrix_rank(singx_hess(x0))}")  # esperado: 3 (singular)
print(f"det(H)     = {np.linalg.det(singx_hess(x0)):.3e}")  # esperado: ≈ 0
# Newton fallará aquí por Hessiana singular
x_sing, _ = newton_method(singx_f, singx_grad, singx_hess, x0, verbose=True)

Caso 1: x0 = [1. 1. 1. 1.]
f(x0)      = 122.0000
||grad||   = 217.2648
rang(H)    = 3
det(H)     = 2.478e-11
k=0: f=1.095194e+04, ||g||=1.077e+04, ||d||=6.207e+00
k=1: f=2.163347e+03, ||g||=3.191e+03, ||d||=1.751e+00
k=2: f=4.273277e+02, ||g||=9.455e+02, ||d||=1.168e+00
k=3: f=8.441042e+01, ||g||=2.801e+02, ||d||=7.784e-01
k=4: f=1.667366e+01, ||g||=8.300e+01, ||d||=5.189e-01
k=5: f=3.293563e+00, ||g||=2.459e+01, ||d||=3.459e-01
k=6: f=6.505804e-01, ||g||=7.287e+00, ||d||=2.306e-01
k=7: f=1.285097e-01, ||g||=2.159e+00, ||d||=1.538e-01
k=8: f=2.538463e-02, ||g||=6.397e-01, ||d||=1.025e-01
k=9: f=5.014248e-03, ||g||=1.895e-01, ||d||=6.833e-02
k=10: f=9.904688e-04, ||g||=5.616e-02, ||d||=4.556e-02
k=11: f=1.956482e-04, ||g||=1.664e-02, ||d||=3.037e-02
k=12: f=3.864655e-05, ||g||=4.931e-03, ||d||=2.025e-02
k=13: f=7.633886e-06, ||g||=1.461e-03, ||d||=1.350e-02
k=14: f=1.507928e-06, ||g||=4.329e-04, ||d||=8.999e-03
k=15: f=2.978624e-07, ||g||=1.283e-04, ||d||=5.999e-03
k=16: f=5.883701e-08,

In [4]:
# ----- Caso 2: Hessiana NO singular en x0 = (1,1,1,0) -----
x0 = np.array([1., 1., 1., 0.])
print(f"\nCaso 2: x0 = {x0}")
print(f"f(x0)      = {singx_f(x0):.4f}")          # esperado: 137.0
print(f"||grad||   = {np.linalg.norm(singx_grad(x0)):.4f}")  # esperado: ≈ 230.92
print(f"rang(H)    = {np.linalg.matrix_rank(singx_hess(x0))}")  # esperado: 4 (no singular)
print(f"autovalores H = {np.linalg.eigvalsh(singx_hess(x0))}")  # todos > 0
print()
x_opt, hist = newton_method(singx_f, singx_grad, singx_hess, x0, tol=1e-10, max_iter=50)
print(f"\nf(x_opt)   = {singx_f(x_opt):.2e}")   # esperado: → 0
print(f"x_opt      = {x_opt}")                    # esperado: → (0,0,0,0)
print(f"Iteraciones: {len(hist['f'])-1}")
# Observación: la primera iteración reduce f de 137 a un valor pequeño;
# a partir de ahí la convergencia es cuadrática hasta llegar cerca de 0,
# donde la Hessiana vuelve a ser singular.


Caso 2: x0 = [1. 1. 1. 0.]
f(x0)      = 137.0000
||grad||   = 230.9199
rang(H)    = 4
autovalores H = [  4.38736996  54.33684452 212.14389897 251.13188655]

k=0: f=2.172840e+00, ||g||=1.697e+01, ||d||=1.339e+00
k=1: f=4.292029e-01, ||g||=5.028e+00, ||d||=3.463e-01
k=2: f=8.478081e-02, ||g||=1.490e+00, ||d||=2.309e-01
k=3: f=1.674683e-02, ||g||=4.414e-01, ||d||=1.539e-01
k=4: f=3.308015e-03, ||g||=1.308e-01, ||d||=1.026e-01
k=5: f=6.534351e-04, ||g||=3.875e-02, ||d||=6.841e-02
k=6: f=1.290736e-04, ||g||=1.148e-02, ||d||=4.560e-02
k=7: f=2.549602e-05, ||g||=3.402e-03, ||d||=3.040e-02
k=8: f=5.036251e-06, ||g||=1.008e-03, ||d||=2.027e-02
k=9: f=9.948150e-07, ||g||=2.987e-04, ||d||=1.351e-02
k=10: f=1.965067e-07, ||g||=8.850e-05, ||d||=9.008e-03
k=11: f=3.881613e-08, ||g||=2.622e-05, ||d||=6.006e-03
k=12: f=7.667384e-09, ||g||=7.769e-06, ||d||=4.004e-03
k=13: f=1.514545e-09, ||g||=2.302e-06, ||d||=2.669e-03
k=14: f=2.991694e-10, ||g||=6.821e-07, ||d||=1.779e-03
k=15: f=5.909519e-11, ||g||